In [ ]:
pip install numpy pandas

<h2>Setup</h2>
Need scikit-learn for this notebook (numpy and pandas are already installed ). To install it in your virtual environment, run:

pip install scikit-learn
<h1>Task 1: regression predicts a number </h1>
Database referencs >> https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset
load_diabetes gives 10 features per patient; the label is a continuous disease-progression score. We fit a line, predict, and measure the average distance between prediction and truth (mean absolute error).

In [ ]:
pip install scikit-learn

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

d = load_diabetes()

d.data  # X : features

In [ ]:
d.target   # y : target variable

In [ ]:
Xr, yr = d.data, d.target
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=0)

reg = LinearRegression().fit(Xr_tr, yr_tr)  # Fitting linear regression model on training data
pred = reg.predict(Xr_te)  # Predicting on test data

print("test MAE (avg distance off):", round(mean_absolute_error(yr_te, pred), 1))
print("one patient  ->  predicted:", round(float(pred[0]), 1),
      " true:", round(float(yr_te[0]), 1),
      " off by:", round(abs(float(pred[0]) - float(yr_te[0])), 1))

<h1>Task 2: classification predicts a category</h1>
load_breast_cancer describes each tumor with 30 numeric features; the label is malignant (0) or benign (1). Two classes, so this is binary classification. 
X (Independant)is what the model sees, y(Dependant) is the answer it must produce.

About data 
Number of Instances: 569
Number of Attributes: 30 numerical attributes used for prediction, along with a class label.
Class Distribution: 212 - Malignant, 357 - Benign

The dataset comprises 30 features, including mean, standard error, and "worst" or largest values, computed for each image. These features encapsulate various aspects of cell nuclei characteristics:

mean radius: Mean of distances from center to points on the perimeter.
mean texture: Standard deviation of gray-scale values.
mean perimeter: Perimeter of the tumor.
mean area: Area of the tumor.
mean smoothness: Variation in radius lengths.
mean compactness: Perimeter^2 / Area - 1.0.
mean concavity: Severity of concave portions of the contour.
mean concave points: Number of concave portions of the contour.
mean symmetry: Symmetry of the cell nuclei.
mean fractal dimension: "Coastline approximation" - 1

In [ ]:
from sklearn.datasets import load_breast_cancer

b = load_breast_cancer()
Xb, yb = b.data, b.target

print("feature matrix X:", Xb.shape)
print("labels y:", yb.shape)
print("classes:", {str(name): int((yb == i).sum()) for i, name in enumerate(b.target_names)})
print("row 0 label:", int(yb[0]), "->", b.target_names[yb[0]])

Split three ways: train, validation, test
Split off the test set first, then split the rest into train and validation. 
The model learns on train
compare choices on validation,
the test set stays sealed until the very end

In [ ]:
X_tv, X_te, y_tv, y_te = train_test_split(
    Xb, yb, test_size=0.2, random_state=0, stratify=yb)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=0, stratify=y_tv)   # 0.25 of 0.8 = 0.2

print("train:", X_tr.shape[0], " validation:", X_val.shape[0], " test:", X_te.shape[0])

Scale on train only >> then choose a model on validation
The scaler is a transform that learns the mean and spread.
 Fit it on the training rows alone, (or test information leaks in).
 Then train two candidates and let validation pick the winner.
 This is the tune loop 

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

scaler = StandardScaler().fit(X_tr)                 # fit on TRAIN only
X_tr_s, X_val_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_val), scaler.transform(X_te)

candidates = {
    "logistic regression": LogisticRegression(max_iter=5000).fit(X_tr_s, y_tr),
    "decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr_s, y_tr),
}
val_acc = {name: accuracy_score(y_val, m.predict(X_val_s)) for name, m in candidates.items()}
for name, a in val_acc.items():
    print(f"validation accuracy  {name:26s} {a:.3f}")

best_name = max(val_acc, key=val_acc.get)
best = candidates[best_name]
print("chosen:", best_name)

In [ ]:
## score the test set once
y_hat = best.predict(X_te_s)
print("TEST accuracy (the number that counts):", round(accuracy_score(y_te, y_hat), 3))

<h1>Better metrics: precision, recall, confusion matrix </h1>
Metrics and scoring: quantifying the quality of predictions
https://scikit-learn.org/stable/modules/model_evaluation.html

<h2> Accuracy hides which mistakes happen. 
Treat malignant (0) as the positive class we must not miss. 
Precision asks how many flagged malignant really were; recall asks how many of the true malignant cases we caught.</h2>

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix
print("precision (malignant):", round(precision_score(y_te, y_hat, pos_label=0), 3))
print("recall    (malignant):", round(recall_score(y_te, y_hat, pos_label=0), 3))
print("confusion matrix (rows = true 0/1, cols = predicted 0/1):")
print(confusion_matrix(y_te, y_hat))

<h1> Multiclass: one of several categories</h1>
Load load_wine has three classes, so the model outputs a score per class and picks the highest. The setup is identical; only the number of classes changed.

In [ ]:
from sklearn.datasets import load_wine
w = load_wine()
Xw, yw = w.data, w.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.2, random_state=0, stratify=yw)
scw = StandardScaler().fit(Xw_tr)
mc = LogisticRegression(max_iter=5000).fit(scw.transform(Xw_tr), yw_tr)

print("classes:", list(map(str, w.target_names)))
print("test accuracy:", round(accuracy_score(yw_te, mc.predict(scw.transform(Xw_te))), 3))

table = {"area": [900, 1500, 2200], "rooms": [2, 3, 4], "price": [240, 355, 512]}
features: ['area', 'rooms']
label: price
task: regression

Predict price using features and labels

In [ ]:
import pandas as pd

# dataset
##Area and Rooms are weight on which price changes
table = {"area": [900, 1500, 2200],
         "rooms": [2, 3, 4],
         "price": [240, 355, 512]}

df = pd.DataFrame(table)
print(df)


In [ ]:
from sklearn.linear_model import LinearRegression

# features and label
X = df[['area', 'rooms']]
y = df['price']

# train model
model = LinearRegression()
model.fit(X, y)

# coefficients
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

In [ ]:
# predict for a new house
new_house = [[800, 2]]  # area=1800, rooms=3
predicted_price = model.predict(new_house)
print("Predicted Price:", predicted_price[0])

The transforms apply, up close
Three transforms show up on almost every dataset. 
Each one learns something from the data it is fit on: the mean and spread, the category list, or the fill value.

In [ ]:
import numpy as np, pandas as pd
from sklearn.impute import SimpleImputer

# scale: put a column on a common scale (mean 0, spread 1)
col = np.array([[10.], [20.], [30.]])
print("scaled:", StandardScaler().fit_transform(col).ravel().round(2))

# encode: turn text categories into 0/1 columns
colors = pd.DataFrame({"color": ["red", "green", "blue", "red"]})
print(pd.get_dummies(colors, columns=["color"]).astype(int).to_string(index=False))

# impute: fill a missing value with the column median
vals = np.array([[1.], [np.nan], [3.], [5.]])
print("imputed:", SimpleImputer(strategy="median").fit_transform(vals).ravel())